# E-Commerce Order Analytics System

Run this notebook from top to bottom. It creates four messy CSV files, cleans them with Pandas, analyzes them with PySpark SQL, and creates a SQLite reporting tool.

## Prerequisites

Install Python 3.10+, Java 17+, and Pandas, NumPy, PySpark, and Jupyter. Java is needed only because PySpark uses the Java Virtual Machine.

pip install pandas numpy pyspark jupyter

In [3]:
from __future__ import annotations
import argparse, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

SEED = 42
RAW_DIR, CLEAN_DIR = Path("data/raw"), Path("data/clean")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. AI generated the required source files

All four files have at least 500 rows. The generator intentionally creates missing customer IDs, malformed dates, invalid emails, untidy product names, and negative quantities for returns.

In [4]:
def generate_raw_data(n_customers=650, n_products=550, n_orders=900, seed=SEED):
    rng = np.random.default_rng(seed)
    today = pd.Timestamp.today().normalize()
    customers = pd.DataFrame(
        {
            "customer_id": [f"C{i:04d}" for i in range(1, n_customers + 1)],
            "customer_name": [f"Customer {i}" for i in range(1, n_customers + 1)],
            "email": [f"customer{i}@example.com" for i in range(1, n_customers + 1)],
            "registration_date": (
                today - pd.to_timedelta(rng.integers(30, 820, n_customers), unit="D")
            ).strftime("%Y-%m-%d"),
            "customer_type": rng.choice(
                ["REGULAR", "PREMIUM", "VIP"], n_customers, p=[0.68, 0.24, 0.08]
            ),
        }
    )
    bad = rng.choice(customers.index, max(1, round(n_customers * 0.02)), replace=False)
    customers.loc[bad, "email"] = "invalid-email"
    cats = ["Electronics", "Clothing", "Home", "Books"]
    names = [
        "Phone Case",
        "Wireless Mouse",
        "Desk Lamp",
        "Python Guide",
        "Running Shoes",
        "Coffee Mug",
    ]
    products = pd.DataFrame(
        {
            "product_id": [f"P{i:04d}" for i in range(1, n_products + 1)],
            "product_name": [names[i % 6] for i in range(n_products)],
            "category": [cats[i % 4] for i in range(n_products)],
            "subcategory": [names[i % 6].split()[-1] for i in range(n_products)],
            "cost_price": np.round(rng.uniform(8, 300, n_products), 2),
        }
    )
    messy = rng.choice(products.index, n_products // 20, replace=False)
    products.loc[messy, "product_name"] = (
        "  " + products.loc[messy, "product_name"].str.upper() + "  "
    )
    dates = pd.Series(today - pd.to_timedelta(rng.integers(0, 730, n_orders), unit="D"))
    orders = pd.DataFrame(
        {
            "order_id": [f"O{i:05d}" for i in range(1, n_orders + 1)],
            "customer_id": rng.choice(customers.customer_id, n_orders),
            "order_date": dates.dt.strftime("%Y-%m-%d 10:00:00"),
            "status": rng.choice(
                ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"],
                n_orders,
                p=[0.1, 0.15, 0.55, 0.08, 0.12],
            ),
            "region_code": rng.choice(["NORTH", "SOUTH", "EAST", "WEST"], n_orders),
        }
    )
    missing = rng.choice(orders.index, round(n_orders * 0.05), replace=False)
    orders.loc[missing, "customer_id"] = None
    malformed = rng.choice(orders.index, n_orders // 35, replace=False)
    orders.loc[malformed, "order_date"] = dates.loc[malformed].dt.strftime("%d-%m-%Y")
    orders.loc[0, "order_date"] = "not-a-date"
    n_items = max(1400, n_orders * 2)
    qty = rng.integers(1, 5, n_items)
    qty[rng.choice(np.arange(n_items), round(n_items * 0.03), replace=False)] *= -1
    items = pd.DataFrame(
        {
            "item_id": [f"I{i:06d}" for i in range(1, n_items + 1)],
            "order_id": rng.choice(orders.order_id, n_items),
            "product_id": rng.choice(products.product_id, n_items),
            "quantity": qty,
            "unit_price": np.round(rng.uniform(12, 520, n_items), 2),
            "discount_percent": rng.choice([0, 5, 10, 15, 20, 25], n_items),
        }
    )
    return customers, products, orders, items


raw_customers, raw_products, raw_orders, raw_order_items = generate_raw_data()
for name, df in zip(
    ["customers", "products", "orders", "order_items"],
    [raw_customers, raw_products, raw_orders, raw_order_items],
):
    assert len(df) >= 500
    df.to_csv(RAW_DIR / f"{name}.csv", index=False)
    print(name, len(df))
assert (
    raw_orders.customer_id.isna().mean() >= 0.04
    and (raw_order_items.quantity < 0).mean() >= 0.02
)

customers 650
products 550
orders 900
order_items 1800


## 2. Cleaning and validation

This first small check demonstrates the test-first idea. It is expected to say that the function does not exist yet.

In [5]:
try:
    clean_orders(pd.DataFrame())
except NameError:
    print("Expected: clean_orders is not defined yet.")

Expected: clean_orders is not defined yet.


In [6]:
def clean_orders(df):
    x = df.copy()
    iso = pd.to_datetime(x.order_date, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    alt = pd.to_datetime(x.order_date, format="%d-%m-%Y", errors="coerce")
    x["parsed"] = iso.fillna(alt)
    today = pd.Timestamp.today().normalize()
    issues = (
        [
            ("orders", r.order_id, "invalid_order_date")
            for _, r in x[x.parsed.isna()].iterrows()
        ]
        + [
            ("orders", r.order_id, "future_order_date")
            for _, r in x[x.parsed > today].iterrows()
        ]
        + [
            ("orders", r.order_id, "missing_customer_id")
            for _, r in x[x.customer_id.isna() | x.customer_id.eq("")].iterrows()
        ]
    )
    return x[x.parsed.notna() & (x.parsed <= today)].drop(columns="order_date").rename(
        columns={"parsed": "order_date"}
    ), pd.DataFrame(issues, columns=["table", "record_id", "issue"])


def clean_products(df):
    x = df.copy()
    x["product_name"] = x.product_name.str.strip().str.title()
    return x


def validate_emails(df):
    return df.loc[
        ~df.email.fillna("").str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), "customer_id"
    ].tolist()


def check_referential_integrity(orders, items):
    return items[~items.order_id.isin(orders.order_id)].copy()


def clean_order_items(df):
    x = df.copy()
    bad = ~x.discount_percent.between(0, 100)
    zero = x.quantity.eq(0)
    issues = pd.DataFrame(
        [("order_items", v, "invalid_discount_percent") for v in x.loc[bad, "item_id"]]
        + [("order_items", v, "zero_quantity") for v in x.loc[zero, "item_id"]],
        columns=["table", "record_id", "issue"],
    )
    return x[~bad & ~zero], issues


clean_orders_df, order_issues = clean_orders(raw_orders)
clean_products_df = clean_products(raw_products)
clean_order_items_df, item_issues = clean_order_items(raw_order_items)
orphans = check_referential_integrity(clean_orders_df, clean_order_items_df)
clean_order_items_df = clean_order_items_df[
    clean_order_items_df.order_id.isin(clean_orders_df.order_id)
]
clean_customers_df = raw_customers.copy()
clean_customers_df["registration_date"] = pd.to_datetime(
    clean_customers_df.registration_date
)
clean_orders_df["order_date"] = pd.to_datetime(clean_orders_df.order_date)
email_issues = pd.DataFrame(
    [("customers", v, "invalid_email") for v in validate_emails(clean_customers_df)],
    columns=["table", "record_id", "issue"],
)
issue_report = pd.concat([order_issues, item_issues, email_issues], ignore_index=True)
for name, df in {
    "customers": clean_customers_df,
    "products": clean_products_df,
    "orders": clean_orders_df,
    "order_items": clean_order_items_df,
    "data_quality_report": issue_report,
}.items():
    df.to_csv(CLEAN_DIR / f"{name}.csv", index=False)
display(issue_report.groupby(["table", "issue"]).size().reset_index(name="count"))

,table,issue,count
0,customers,invalid_email,13
1,orders,future_order_date,1
2,orders,invalid_order_date,1
3,orders,missing_customer_id,45


## 3. PySpark SQL analysis

Spark runs locally. The queries include joins, aggregations, CTEs, window functions, cohort retention, and product pairs.

In [7]:
spark = SparkSession.builder.master("local[2]").appName("OrderAnalytics").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
for name, df in {
    "customers": clean_customers_df,
    "products": clean_products_df,
    "orders": clean_orders_df,
    "order_items": clean_order_items_df,
}.items():
    spark.createDataFrame(df).createOrReplaceTempView(name)
spark.sql(
    "CREATE OR REPLACE TEMP VIEW valid_sales AS SELECT oi.*,o.customer_id,o.order_date,o.status,o.region_code,p.product_name,p.category,oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0) revenue FROM order_items oi JOIN orders o ON oi.order_id=o.order_id JOIN products p ON oi.product_id=p.product_id"
)


def show_query(title, sql):
    print(title)
    display(spark.sql(sql).limit(10).toPandas())


queries = {
    "1 Revenue per category": "SELECT category,SUM(revenue) total_revenue FROM valid_sales GROUP BY category",
    "2 Top customers": "SELECT customer_id,SUM(revenue) total_value FROM valid_sales WHERE customer_id IS NOT NULL GROUP BY customer_id ORDER BY total_value DESC LIMIT 10",
    "3 Last 12 months": "WITH m AS (SELECT MAX(order_date) d FROM orders) SELECT DATE_TRUNC('month',o.order_date) month,COUNT(*) order_count FROM orders o CROSS JOIN m WHERE o.order_date>=ADD_MONTHS(m.d,-11) GROUP BY DATE_TRUNC('month',o.order_date)",
    "4 Never delivered": "SELECT customer_id FROM orders WHERE customer_id IS NOT NULL GROUP BY customer_id HAVING SUM(CASE WHEN status='DELIVERED' THEN 1 ELSE 0 END)=0",
    "5 More returns": "SELECT product_id,SUM(CASE WHEN quantity<0 THEN ABS(quantity) ELSE 0 END) returns,SUM(CASE WHEN quantity>0 THEN quantity ELSE 0 END) purchases FROM order_items GROUP BY product_id HAVING returns>purchases",
    "6 Return rate": "SELECT category,100.0*SUM(CASE WHEN quantity<0 THEN ABS(quantity) ELSE 0 END)/SUM(ABS(quantity)) return_rate FROM valid_sales GROUP BY category",
    "7 Running total": "WITH d AS (SELECT region_code,CAST(order_date AS DATE) order_date,SUM(revenue) daily_revenue FROM valid_sales GROUP BY region_code,CAST(order_date AS DATE)) SELECT *,SUM(daily_revenue) OVER(PARTITION BY region_code ORDER BY order_date) running_total FROM d",
    "8 Dense rank": "WITH x AS (SELECT category,product_name,SUM(revenue) total_revenue FROM valid_sales GROUP BY category,product_name) SELECT *,DENSE_RANK() OVER(PARTITION BY category ORDER BY total_revenue DESC) rank_in_category FROM x",
    "9 Lag gap": "WITH x AS (SELECT customer_id,order_date,LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date) previous_order_date FROM orders WHERE customer_id IS NOT NULL) SELECT *,DATEDIFF(order_date,previous_order_date) days_gap FROM x",
    "10 CTE segments": "WITH m AS (SELECT customer_id,DATE_TRUNC('month',order_date) month,SUM(revenue) revenue FROM valid_sales WHERE customer_id IS NOT NULL GROUP BY customer_id,DATE_TRUNC('month',order_date)),s AS (SELECT *,CASE WHEN revenue>10000 THEN 'High' WHEN revenue>=5000 THEN 'Medium' ELSE 'Low' END segment FROM m) SELECT month,segment,COUNT(*) customers FROM s GROUP BY month,segment",
    "11 NTILE": "WITH x AS (SELECT customer_id,SUM(revenue) total_value FROM valid_sales WHERE customer_id IS NOT NULL GROUP BY customer_id) SELECT *,NTILE(4) OVER(ORDER BY total_value DESC) quartile FROM x",
    "12 YoY": "WITH m AS (SELECT YEAR(order_date) year,MONTH(order_date) month,SUM(revenue) revenue FROM valid_sales GROUP BY YEAR(order_date),MONTH(order_date)) SELECT *,LAG(revenue) OVER(PARTITION BY month ORDER BY year) prev_year_revenue FROM m",
    "13 First last": "WITH r AS (SELECT customer_id,category,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date) a,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC) b FROM valid_sales WHERE customer_id IS NOT NULL) SELECT customer_id,MAX(CASE WHEN a=1 THEN category END) first_category,MAX(CASE WHEN b=1 THEN category END) last_category FROM r GROUP BY customer_id",
    "14 Cumulative": "WITH x AS (SELECT customer_id,SUM(revenue) revenue FROM valid_sales WHERE customer_id IS NOT NULL GROUP BY customer_id) SELECT *,SUM(revenue) OVER(ORDER BY revenue DESC) cumulative_revenue,100.0*SUM(revenue) OVER(ORDER BY revenue DESC)/SUM(revenue) OVER() cumulative_percent FROM x",
    "15 Cohort": "WITH c AS (SELECT customer_id,DATE_TRUNC('month',registration_date) cohort FROM customers),a AS (SELECT c.cohort,FLOOR(MONTHS_BETWEEN(DATE_TRUNC('month',o.order_date),c.cohort)) month_number,c.customer_id FROM c JOIN orders o ON c.customer_id=o.customer_id) SELECT cohort,month_number,COUNT(DISTINCT customer_id) active_customers FROM a WHERE month_number BETWEEN 0 AND 3 GROUP BY cohort,month_number",
    "16 Product pairs": "SELECT a.product_id product_a,b.product_id product_b,COUNT(*) times_bought_together FROM order_items a JOIN order_items b ON a.order_id=b.order_id AND a.product_id<b.product_id GROUP BY a.product_id,b.product_id ORDER BY times_bought_together DESC",
}
for title, sql in queries.items():
    show_query(title, sql)
for sql in queries.values():
    spark.sql(sql).limit(1).collect()
print("All 16 SQL queries executed successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/10 00:57:48 WARN Utils: Your hostname, Kshitizs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/08/10 00:57:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 00:57:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/10 00:57:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


1 Revenue per category


,category,total_revenue
0,Home,251365.1105
1,Electronics,242136.2970
2,Clothing,244552.2265
3,Books,253552.6730


2 Top customers


,customer_id,total_value
0,C0584,10436.0785
1,C0030,8623.8350
2,C0372,7884.6620
3,C0412,7695.4105
4,C0497,7275.7990
5,C0292,7239.6210
6,C0269,7004.6790
7,C0045,6934.8370
8,C0099,6741.2750
9,C0492,6666.0280


...

15 Cohort


,cohort,month_number,active_customers
0,2025-08-01,2,3
1,2024-10-01,2,1
2,2025-04-01,0,3
3,2025-08-01,1,2
4,2025-01-01,2,1
5,2024-11-01,2,2
6,2024-09-01,3,1
7,2025-11-01,2,3
8,2024-10-01,0,1
9,2025-03-01,2,3


16 Product pairs


,product_a,product_b,times_bought_together
0,P0036,P0485,2
1,P0396,P0411,2
2,P0092,P0437,2
3,P0279,P0425,2
4,P0171,P0254,2
5,P0135,P0378,2
6,P0133,P0256,2
7,P0285,P0355,2
8,P0289,P0506,2
9,P0034,P0148,2


All 16 SQL queries executed successfully.


## 4. SQLite reporting tool

In [8]:
def build_reporting_database(customers, products, orders, items):
    c = sqlite3.connect(":memory:")
    c.row_factory = sqlite3.Row
    for table, df in {
        "customers": customers,
        "products": products,
        "orders": orders,
        "order_items": items,
    }.items():
        df.to_sql(table, c, index=False)
    return c


def run_report(argv, conn):
    p = argparse.ArgumentParser(add_help=False, exit_on_error=False)
    p.add_argument("--report", choices=["daily", "weekly", "monthly"], required=True)
    p.add_argument("--start", required=True)
    p.add_argument("--end", required=True)
    try:
        a = p.parse_args(argv)
        start, end = pd.Timestamp(a.start), pd.Timestamp(a.end)
    except (ValueError, argparse.ArgumentError) as e:
        raise ValueError(
            "Use --report daily|weekly|monthly --start YYYY-MM-DD --end YYYY-MM-DD."
        ) from e
    if start > end:
        raise ValueError("Start date must be on or before end date.")
    rev = "oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)"
    summary = dict(
        conn.execute(
            f"SELECT COUNT(DISTINCT o.order_id) total_orders,ROUND(COALESCE(SUM({rev}),0),2) total_revenue,COUNT(DISTINCT o.customer_id) unique_customers FROM orders o LEFT JOIN order_items oi ON o.order_id=oi.order_id WHERE DATE(o.order_date) BETWEEN DATE(?) AND DATE(?)",
            (start.date(), end.date()),
        ).fetchone()
    )
    top = [
        dict(x)
        for x in conn.execute(
            f"SELECT p.product_name,ROUND(SUM({rev}),2) revenue FROM orders o JOIN order_items oi ON o.order_id=oi.order_id JOIN products p ON oi.product_id=p.product_id WHERE DATE(o.order_date) BETWEEN DATE(?) AND DATE(?) GROUP BY p.product_name ORDER BY revenue DESC LIMIT 3",
            (start.date(), end.date()),
        ).fetchall()
    ]
    return {"report_type": a.report, "summary": summary, "top_products": top}


report_conn = build_reporting_database(
    clean_customers_df, clean_products_df, clean_orders_df, clean_order_items_df
)
report = run_report(
    [
        "--report",
        "monthly",
        "--start",
        "2025-01-01",
        "--end",
        str(clean_orders_df.order_date.max().date()),
    ],
    report_conn,
)
print(report)

{'report_type': 'monthly', 'summary': {'total_orders': 729, 'total_revenue': 817636.79, 'unique_customers': 415}, 'top_products': [{'product_name': 'Running Shoes', 'revenue': 149527.97}, {'product_name': 'Python Guide', 'revenue': 148772.74}, {'product_name': 'Desk Lamp', 'revenue': 140380.12}]}


## 5. Edge-case tests

In [9]:
def test_edge_cases():
    assert (
        len(
            check_referential_integrity(
                pd.DataFrame({"order_id": ["O1"]}), pd.DataFrame({"order_id": ["O9"]})
            )
        )
        == 1
    )
    _, x = clean_order_items(
        pd.DataFrame(
            {
                "item_id": ["I"],
                "order_id": ["O"],
                "product_id": ["P"],
                "quantity": [1],
                "unit_price": [1],
                "discount_percent": [101],
            }
        )
    )
    assert x.issue.tolist() == ["invalid_discount_percent"]
    _, x = clean_order_items(
        pd.DataFrame(
            {
                "item_id": ["I"],
                "order_id": ["O"],
                "product_id": ["P"],
                "quantity": [0],
                "unit_price": [1],
                "discount_percent": [0],
            }
        )
    )
    assert x.issue.tolist() == ["zero_quantity"]
    future = (pd.Timestamp.today() + pd.Timedelta(days=2)).strftime("%Y-%m-%d %H:%M:%S")
    _, x = clean_orders(
        pd.DataFrame(
            {
                "order_id": ["O"],
                "customer_id": ["C"],
                "order_date": [future],
                "status": ["PLACED"],
                "region_code": ["NORTH"],
            }
        )
    )
    assert "future_order_date" in x.issue.tolist()
    try:
        run_report(
            ["--report", "yearly", "--start", "2025-01-01", "--end", "2025-01-02"],
            report_conn,
        )
        raise AssertionError()
    except ValueError:
        pass


test_edge_cases()
print("All validation checks passed. Run spark.stop() when finished.")

All validation checks passed. Run spark.stop() when finished.
